# LightGBM Regression – Product Sales Prediction
Đọc dữ liệu → làm sạch → lưu dữ liệu sạch → chia Train/Test 80/20 → huấn luyện LightGBM → đánh giá → trực quan hóa.


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from lightgbm_regression import LightGBMRegression
from regression_metrics import print_regression_metrics


In [ ]:
df = pd.read_csv("data/raw/market_data.csv")
print("===== DỮ LIỆU GỐC =====")
display(df.head())
print("Shape:", df.shape)


In [ ]:
print("===== THÔNG TIN DATASET =====")
df.info()
print("\n===== GIÁ TRỊ THIẾU =====")
print(df.isnull().sum())
print("\nSố dòng trùng:", df.duplicated().sum())


In [ ]:
df = df.drop_duplicates().copy()
for column in df.columns:
    if pd.api.types.is_numeric_dtype(df[column]):
        df[column] = df[column].fillna(df[column].median())
    else:
        mode_values = df[column].mode()
        if not mode_values.empty:
            df[column] = df[column].fillna(mode_values.iloc[0])
print("Giá trị thiếu sau xử lý:")
print(df.isnull().sum())
print("Số dòng trùng sau xử lý:", df.duplicated().sum())


In [ ]:
categorical_columns = df.select_dtypes(include=["object", "category"]).columns.tolist()
print("Các cột categorical:", categorical_columns)
if categorical_columns:
    df = pd.get_dummies(df, columns=categorical_columns, drop_first=False, dtype=int)
    print("Đã mã hóa categorical thành số.")
else:
    print("Dataset chỉ có dữ liệu số, không cần mã hóa.")


In [ ]:
os.makedirs("data/processed", exist_ok=True)
clean_path = "data/processed/regression_cleaned.csv"
df.to_csv(clean_path, index=False)
print("Đã lưu:", clean_path)


In [ ]:
TARGET = "Sale"
X_features = df.drop(columns=[TARGET])
y_label = df[TARGET]
print("Features:", X_features.columns.tolist())
print("Target:", TARGET)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_label, test_size=0.2, random_state=101
)
print("X_train shape :", X_train.shape)
print("y_train shape :", y_train.shape)
print("=" * 20)
print("X_test shape  :", X_test.shape)
print("y_test shape  :", y_test.shape)


In [ ]:
model = LightGBMRegression(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    random_state=42
)
model.fit(X_train, y_train)
print("Huấn luyện mô hình thành công.")


In [ ]:
y_pred = model.predict(X_test)
print("Số giá trị dự đoán:", len(y_pred))
print("5 dự đoán đầu:", y_pred[:5])


In [ ]:
metrics = print_regression_metrics(y_test, y_pred)


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.7)
min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())
plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")
plt.xlabel("Actual Sale")
plt.ylabel("Predicted Sale")
plt.title("LightGBM Regression - Actual vs Predicted")
plt.tight_layout()
plt.show()


In [ ]:
importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.get_feature_importance()
}).sort_values(by="Importance", ascending=False)
display(importance)
plt.figure(figsize=(8, 6))
plt.barh(importance["Feature"], importance["Importance"])
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.title("LightGBM Feature Importance")
plt.tight_layout()
plt.show()
